<a href="https://colab.research.google.com/github/Efe-Godson/3mtt-stage2-analysis/blob/main/notebooks/deliverable_2_analysis_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3MTT STAGE 2: Data Analysis Notebook

**TABLE OF CONTENT**

# 1. ENVIRONMENT SETUP AND LIBRARY IMPORTS

In [26]:
# CORE LIBRARIES

# Data manipulation
import pandas as pd
import numpy as np

# SQLite database connection
import sqlite3

# Date handling
from datetime import datetime

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

#Regular Expression operator
import re

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

#2. DATA LOADING

Raw datasets are loaded directly from the GitHub repository

In [16]:
# Loading raw data

# Fellows cohort dataset
fellows = pd.read_csv("https://raw.githubusercontent.com/Efe-Godson/3mtt-stage2-analysis/main/raw_data/fellows_cohort.csv")

# Reflection survey dataset
reflection = pd.read_excel("https://raw.githubusercontent.com/Efe-Godson/3mtt-stage2-analysis/main/raw_data/reflection_survey.xlsx", skiprows=1)

# Employer engagement dataset
employer = pd.read_csv("https://raw.githubusercontent.com/Efe-Godson/3mtt-stage2-analysis/main/raw_data/employer_engagement.csv")

# ALC weekly operational logs
alc_logs = pd.read_csv("https://raw.githubusercontent.com/Efe-Godson/3mtt-stage2-analysis/main/raw_data/alc_weekly_log.csv")

print("All datasets loaded successfully.")

All datasets loaded successfully.


## 2.1 STRUCTURAL ISSUE - Reflection Survey File

The reflection survey dataset included an extra merged row above the actual column headers.

The file was loaded using the `skiprows=1` parameter in `pandas.read_excel()` so the correct headers could be captured properly.

This prevented column alignment issues later in the analysis.

#3. INITIAL DATA FAMILIARIZATION

## 3.1 STRUCTURAL INSPECTION

In [17]:
from IPython.display import display, HTML


# Store datasets in a dictionary
datasets = {
    "Fellows Cohort": fellows,
    "Reflection Survey": reflection,
    "Employer Engagement": employer,
    "ALC Weekly Logs": alc_logs
}


# Start responsive grid container
html_content = """
<div style="
    display:grid;
    grid-template-columns:repeat(auto-fit, minmax(420px, 1fr));
    gap:20px;
    width:100%;
">
"""


# Generate overview cards
for name, df in datasets.items():

    html_content += f"""

    <div style="
        border:1px solid #d9d9d9;
        border-radius:12px;
        padding:18px;
        background-color:white;
        box-shadow:2px 2px 10px rgba(0,0,0,0.08);
        overflow-x:auto;
    ">

        <h2 style="color:#1f4e79; margin-top:0;">
            {name}
        </h2>

        <hr>

        <p><strong>Rows:</strong> {df.shape[0]}</p>
        <p><strong>Columns:</strong> {df.shape[1]}</p>

        <h4>Column Names</h4>

        <div style="
            background-color:#f7f7f7;
            padding:10px;
            border-radius:6px;
            font-size:13px;
            max-height:120px;
            overflow-y:auto;
        ">
            {", ".join(df.columns)}
        </div>

        <h4 style="margin-top:20px;">
            Data Types
        </h4>

        {df.dtypes.to_frame(name='dtype').to_html(index=True)}

    </div>

    """


# Close grid container
html_content += "</div>"


# Render overview dashboard
display(HTML(html_content))

,dtype
fellow_id,object
first_name,object
last_name,object
state,object
alc_code,object
cohort_number,int64
track,object
enrollment_date,object
completion_status,object
certification_status,object


## 3.2 SAMPLE RECORD INSPECTION

In [18]:
# Display sample records from each dataset
for name, df in datasets.items():

    print(f"\n{'='*60}")
    print(f"{name.upper()} - SAMPLE RECORDS")
    print(f"{'='*60}")

    display(df.head())


FELLOWS COHORT - SAMPLE RECORDS


,fellow_id,first_name,last_name,state,alc_code,cohort_number,track,enrollment_date,completion_status,certification_status
0,3MTT-F00001,Halima,Musa,niger,ALC-013,3,AI/ML,2024-07-27,complete,certified
1,3MTT-F00002,Grace,Okafor,Ebonyi,ALC-037,4,Cybersecurity,2024-02-20,complete,certified
2,3MTT-F00003,Yusuf,Yakubu,Edo,ALC-041,4,Data Analysis,2024-10-26,complete,certified
3,3MTT-F00004,Ekene,Afolabi,Kaduna,ALC-005,4,AI/ML,2024-08-07,complete,certified
4,3MTT-F00005,Grace,Abubakar,Jigawa,ALC-031,4,Cloud Computing,2024-09-05,complete,certified



REFLECTION SURVEY - SAMPLE RECORDS


,fellow_id,week,response_timestamp,survey_score,phone_number,email
0,3MTT-F00001,1,2024-02-05 16:00:00,2.2,+2348165956164,3mtt.f00001@gmail.com
1,3MTT-F00001,2,2024-02-14 13:00:00,4.9,2347067490250,3mtt.f00001@gmail.com
2,3MTT-F00001,4,2024-03-01 11:00:00,4.4,2349058953690,3mtt.f00001@gmail.com
3,3MTT-F00001,5,2024-03-09 09:00:00,4.9,0816-989-8681,3mtt.f00001@yahoo.com
4,3MTT-F00001,7,2024-03-23 16:00:00,3.6,+2347064394353,3mtt.f00001@yahoo.com



EMPLOYER ENGAGEMENT - SAMPLE RECORDS


,employer_name,sector,state,geopolitical_zone,engagement_type,engagement_date,alc_code,fellows_referred
0,Nestle Nigeria,FMCG,Niger,North-Central,webinar,2024-12-31,ALC-045,0
1,Nestle Nigeria,FMCG,Lagos,South-West,fair,2024-03-10,ALC-016,4
2,Tek Experts,Technology,Borno,North-East,placement,2024-10-08,ALC-030,13
3,Inlaks,Technology,Ekiti,South-West,bootcamp,2024-09-29,ALC-040,0
4,Helium Health,HealthTech,Zamfara,North-West,placement,2024-08-03,ALC-037,13



ALC WEEKLY LOGS - SAMPLE RECORDS


,alc_code,week_number,state,geopolitical_zone,fellows_present,sessions_held,facilitator_name,data_quality_flag
0,ALC-001,1,Adamawa,North-East,15,5,Fatima Aliyu,OK
1,ALC-001,2,Adamawa,North-East,7,2,Fatima Aliyu,LOW_ATTENDANCE
2,ALC-001,3,Adamawa,North-East,10,4,Fatima Aliyu,OK
3,ALC-001,4,Adamawa,North-East,9,3,Fatima Aliyu,OK
4,ALC-001,5,Adamawa,North-East,14,3,Fatima Aliyu,OK


##3.3 MISSING VALUE OVERVIEW

In [19]:
# Display missing value summaries across datasets
for name, df in datasets.items():

    missing_summary = pd.DataFrame({
        "Missing Values": df.isnull().sum(),
        "Missing Percentage": (
            df.isnull().mean() * 100
        ).round(2)
    })

    missing_summary = (
        missing_summary[
            missing_summary["Missing Values"] > 0
        ]
        .sort_values(
            by="Missing Values",
            ascending=False
        )
    )


    # Display only datasets containing missing values
    if not missing_summary.empty:

        print(f"\n{name}")

        display(
            missing_summary.style
            .background_gradient(
                cmap="Oranges",
                subset=["Missing Percentage"]
            )
            .format({
                "Missing Percentage": "{:.2f}%"
            })
        )


Fellows Cohort


,Missing Values,Missing Percentage
alc_code,47,4.00%



Employer Engagement


,Missing Values,Missing Percentage
alc_code,22,5.12%



ALC Weekly Logs


,Missing Values,Missing Percentage
alc_code,20,3.02%


## 3.4 DUPLICATE RECORD OVERVIEW

In [20]:
# Analyze duplicate records across datasets
for name, df in datasets.items():

    duplicate_count = df.duplicated().sum()

    duplicate_percentage = round(
        (duplicate_count / len(df)) * 100,
        2
    )


    # Display duplicate summary for each dataset
    duplicate_summary = pd.DataFrame({
        "Duplicate Records": [duplicate_count],
        "Duplicate Percentage": [duplicate_percentage]
    })

    print(f"\n{name}")

    display(
        duplicate_summary.style
        .background_gradient(
            cmap="Reds",
            subset=["Duplicate Percentage"]
        )
        .format({
            "Duplicate Percentage": "{:.2f}%"
        })
    )


Fellows Cohort


,Duplicate Records,Duplicate Percentage
0,20,1.70%



Reflection Survey


,Duplicate Records,Duplicate Percentage
0,30,0.30%



Employer Engagement


,Duplicate Records,Duplicate Percentage
0,5,1.16%



ALC Weekly Logs


,Duplicate Records,Duplicate Percentage
0,15,2.26%


## 3.6 UNIQUE VALUE INSPECTION

In [21]:
# Inspect categorical uniqueness across datasets
for name, df in datasets.items():

    print(f"\n{name}")


    # Select categorical columns for inspection
    categorical_columns = df.select_dtypes(
        include="object"
    ).columns


    # Generate unique value summary
    unique_summary = pd.DataFrame({
        "Unique Values": [
            df[col].nunique()
            for col in categorical_columns
        ]
    },
    index=categorical_columns)


    # Display categorical uniqueness overview
    display(
        unique_summary.style
        .background_gradient(
            cmap="Blues"
        )
    )


Fellows Cohort


,Unique Values
fellow_id,1150
first_name,25
last_name,25
state,103
alc_code,54
track,8
enrollment_date,286
completion_status,2
certification_status,2



Reflection Survey


,Unique Values
fellow_id,1150
response_timestamp,1092
phone_number,9944
email,2243



Employer Engagement


,Unique Values
employer_name,29
sector,10
state,62
geopolitical_zone,6
engagement_type,5
engagement_date,257
alc_code,54



ALC Weekly Logs


,Unique Values
alc_code,54
state,29
geopolitical_zone,6
facilitator_name,15
data_quality_flag,2


## 3.7 NUMERICAL VARIABLE SUMMARY

In [22]:
# Inspect numerical distributions across datasets
for name, df in datasets.items():


    # Select numerical columns
    numerical_columns = df.select_dtypes(
        include=["int64", "float64"]
    )


    # Skip datasets without numerical columns
    if not numerical_columns.empty:

        print(f"\n{name}")


        # Generate descriptive statistics
        numerical_summary = (
            numerical_columns
            .describe()
            .T
            .round(2)
        )


        # Display numerical summary table
        display(
            numerical_summary.style
            .background_gradient(
                cmap="Greens"
            )
        )


Fellows Cohort


,count,mean,std,min,25%,50%,75%,max
cohort_number,1175.000000,3.520000,0.500000,3.000000,3.000000,4.000000,4.000000,4.000000



Reflection Survey


,count,mean,std,min,25%,50%,75%,max
week,9974.000000,6.510000,3.470000,1.000000,3.000000,7.000000,10.000000,12.000000
survey_score,9974.000000,3.510000,0.870000,2.000000,2.800000,3.500000,4.300000,5.000000



Employer Engagement


,count,mean,std,min,25%,50%,75%,max
fellows_referred,430.000000,5.040000,8.610000,0.000000,0.000000,0.000000,7.000000,30.000000



ALC Weekly Logs


,count,mean,std,min,25%,50%,75%,max
week_number,663.000000,7.170000,8.720000,1.000000,3.500000,7.000000,9.000000,99.000000
fellows_present,663.000000,13.380000,7.590000,2.000000,8.000000,12.000000,17.500000,39.000000
sessions_held,663.000000,3.520000,1.120000,2.000000,2.000000,4.000000,4.500000,5.000000


## 3.8 TEMPORAL VARIABLE INSPECTION

In [23]:
# Inspect date and time-related columns across datasets
for name, df in datasets.items():

    # Identify columns containing date or time keywords
    datetime_columns = [
        col for col in df.columns
        if "date" in col.lower()
        or "time" in col.lower()
    ]


    # Skip datasets without temporal columns
    if len(datetime_columns) > 0:

        print(f"\n{name}")


        # Create a quick preview of temporal fields
        temporal_summary = pd.DataFrame({
            "Column": datetime_columns,
            "Sample Values": [
                df[col]
                .dropna()
                .astype(str)
                .unique()[:3]
                for col in datetime_columns
            ]
        })


        # Display temporal inspection summary
        display(
            temporal_summary.style
            .background_gradient(cmap="Purples")
        )


Fellows Cohort


,Column,Sample Values
0,enrollment_date,['2024-07-27' '2024-02-20' '2024-10-26']



Reflection Survey


,Column,Sample Values
0,response_timestamp,['2024-02-05 16:00:00' '2024-02-14 13:00:00' '2024-03-01 11:00:00']



Employer Engagement


,Column,Sample Values
0,engagement_date,['2024-12-31' '2024-03-10' '2024-10-08']


The temporal fields loaded successfully across the datasets and generally followed a consistent date structure.

The reflection survey dataset contained timestamp values, while the remaining datasets used date-only fields.

No obvious formatting issues were identified during the initial inspection.

# 4.  DATA CLEANING AND STANDARDISATION

## 4.1 STANDARDIZE STRUCTURAL CONSISTENCY

In [24]:
# Create copies of the raw datasets before transformation
fellows_clean = fellows.copy()
reflection_clean = reflection.copy()
employer_clean = employer.copy()
alc_logs_clean = alc_logs.copy()


# Store cleaned datasets in a dictionary for easier iteration
cleaned_datasets = {
    "Fellows Cohort": fellows_clean,
    "Reflection Survey": reflection_clean,
    "Employer Engagement": employer_clean,
    "ALC Weekly Logs": alc_logs_clean
}


# ---------------------------------------------------------
# Standardize temporal columns across datasets
# ---------------------------------------------------------


# Convert enrollment dates into datetime format
fellows_clean["enrollment_date"] = pd.to_datetime(
    fellows_clean["enrollment_date"],
    errors="coerce"
)


# Convert survey timestamps into datetime format
reflection_clean["response_timestamp"] = pd.to_datetime(
    reflection_clean["response_timestamp"],
    errors="coerce"
)


# Convert employer engagement dates into datetime format
employer_clean["engagement_date"] = pd.to_datetime(
    employer_clean["engagement_date"],
    errors="coerce"
)


# ---------------------------------------------------------
# Validate datetime parsing results
# ---------------------------------------------------------


# Store temporal validation summaries
temporal_validation = pd.DataFrame({

    "Dataset": [
        "Fellows Cohort",
        "Reflection Survey",
        "Employer Engagement"
    ],

    "Temporal Column": [
        "enrollment_date",
        "response_timestamp",
        "engagement_date"
    ],

    "Null Values After Parsing": [
        fellows_clean["enrollment_date"].isnull().sum(),
        reflection_clean["response_timestamp"].isnull().sum(),
        employer_clean["engagement_date"].isnull().sum()
    ]

})


# Display datetime validation results
display(
    temporal_validation.style
    .background_gradient(
        cmap="Purples",
        subset=["Null Values After Parsing"]
    )
)


# ---------------------------------------------------------
# Standardize categorical location fields
# ---------------------------------------------------------


# Normalize state formatting in fellows dataset
fellows_clean["state"] = (
    fellows_clean["state"]
    .str.strip()
    .str.title()
)


# Normalize state formatting in employer dataset
employer_clean["state"] = (
    employer_clean["state"]
    .str.strip()
    .str.title()
)


# Normalize state formatting in ALC logs dataset
alc_logs_clean["state"] = (
    alc_logs_clean["state"]
    .str.strip()
    .str.title()
)


# ---------------------------------------------------------
# Recheck categorical consistency after standardization
# ---------------------------------------------------------


# Compare unique state counts after cleaning
state_validation = pd.DataFrame({

    "Dataset": [
        "Fellows Cohort",
        "Employer Engagement",
        "ALC Weekly Logs"
    ],

    "Unique States": [
        fellows_clean["state"].nunique(),
        employer_clean["state"].nunique(),
        alc_logs_clean["state"].nunique()
    ]

})


# Display state validation summary
display(
    state_validation.style
    .background_gradient(
        cmap="Blues",
        subset=["Unique States"]
    )
)


# ---------------------------------------------------------
# Inspect standardized temporal samples
# ---------------------------------------------------------


# Display cleaned temporal samples for validation
temporal_samples = pd.DataFrame({

    "Enrollment Date": fellows_clean["enrollment_date"].head(3),

    "Response Timestamp": (
        reflection_clean["response_timestamp"].head(3)
        .reset_index(drop=True)
    ),

    "Engagement Date": (
        employer_clean["engagement_date"].head(3)
        .reset_index(drop=True)
    )

})


# Render temporal validation samples
display(temporal_samples)

,Dataset,Temporal Column,Null Values After Parsing
0,Fellows Cohort,enrollment_date,0
1,Reflection Survey,response_timestamp,0
2,Employer Engagement,engagement_date,0


,Dataset,Unique States
0,Fellows Cohort,37
1,Employer Engagement,37
2,ALC Weekly Logs,29


,Enrollment Date,Response Timestamp,Engagement Date
0,2024-07-27,2024-02-05 16:00:00,2024-12-31
1,2024-02-20,2024-02-14 13:00:00,2024-03-10
2,2024-10-26,2024-03-01 11:00:00,2024-10-08


The temporal fields were successfully converted into datetime format without any parsing issues. This confirms that the date-related fields were structurally consistent across the datasets.

State names were also standardized to improve categorical consistency. Earlier variations such as `niger`, `IMO`, and `Lagos` were normalized into a consistent format, making the location fields more reliable for downstream analysis and grouping operations.

## 4.2 CLEAN IDENTITY & CONTACT FIELDS

In [25]:
# Inspect raw phone number patterns

# Create a working copy of the phone number column
phone_audit = reflection_clean.copy()


# Store original phone numbers for comparison
phone_audit["phone_original"] = (
    phone_audit["phone_number"]
    .astype(str)
)


# Extract only numeric characters
phone_audit["phone_digits"] = (
    phone_audit["phone_original"]
    .str.replace(r"\D", "", regex=True)
)


# Analyze phone number lengths before cleaning
phone_length_before = (
    phone_audit["phone_digits"]
    .str.len()
    .value_counts()
    .sort_index()
    .reset_index()
)

phone_length_before.columns = [
    "Phone Length",
    "Record Count"
]


# Display raw phone structure distribution
display(
    phone_length_before.style
    .background_gradient(cmap="Oranges")
)


# Standardize Nigerian phone number format

# Define phone number cleaning function
def standardize_nigerian_phone(phone):

    # Convert values to string format
    phone = str(phone)


    # Remove non-numeric characters
    phone = re.sub(r"\D", "", phone)


    # Remove extra leading international zeros
    if phone.startswith("00"):
        phone = phone[2:]


    # Convert local Nigerian format
    if len(phone) == 11 and phone.startswith("0"):
        phone = "234" + phone[1:]


    # Convert short 10-digit numbers
    elif len(phone) == 10:
        phone = "234" + phone


    # Validate final structure
    if len(phone) == 13 and phone.startswith("234"):
        return phone


    # Return invalid values as NaN
    return np.nan


# Apply phone standardization
reflection_clean["phone_clean"] = (
    reflection_clean["phone_number"]
    .apply(standardize_nigerian_phone)
)


# Validate phone cleaning results

# Count valid standardized numbers
valid_phone_count = (
    reflection_clean["phone_clean"]
    .notnull()
    .sum()
)


# Count invalid phone records
invalid_phone_count = (
    reflection_clean["phone_clean"]
    .isnull()
    .sum()
)


# Compare unique phone formats before and after cleaning
phone_validation = pd.DataFrame({

    "Metric": [
        "Valid Standardized Numbers",
        "Invalid Remaining Values",
        "Unique Raw Phone Values",
        "Unique Clean Phone Values"
    ],

    "Count": [
        valid_phone_count,
        invalid_phone_count,
        reflection_clean["phone_number"].nunique(),
        reflection_clean["phone_clean"].nunique()
    ]

})


# Display phone validation summary
display(
    phone_validation.style
    .background_gradient(
        cmap="Greens",
        subset=["Count"]
    )
)


# Compare raw and cleaned phone samples

# Display before-and-after phone comparison
phone_comparison = reflection_clean[[
    "phone_number",
    "phone_clean"
]].head(15)


# Render phone transformation samples
display(phone_comparison)

,Phone Length,Record Count
0,10,1686
1,11,3304
2,12,1694
3,13,3290


NameError: name 're' is not defined

The phone number field contained several inconsistent formats, including local Nigerian numbers, values with symbols, and numbers with extra leading zeros.

Numbers beginning with `0` were converted into the international Nigerian format by replacing the leading zero with `234`. Symbols such as `+` and `-` were removed, while values beginning with `00` were corrected before standardization.

After cleaning, all phone numbers were successfully converted into the required `234XXXXXXXXXX` structure, making the field more consistent for identity matching and deduplication.

## 4.3 HANDLE MISSING DATA

In [ ]:

# Inspect missing alc_code values across datasets
alc_missing_summary = pd.DataFrame({

    "Dataset": [
        "Fellows Cohort",
        "Employer Engagement"
    ],

    "Missing alc_code Values": [
        fellows_clean["alc_code"].isnull().sum(),
        employer_clean["alc_code"].isnull().sum()
    ]

})


# Display missing alc_code summary
display(
    alc_missing_summary.style
    .background_gradient(
        cmap="Oranges",
        subset=["Missing alc_code Values"]
    )
)


# Review records with missing alc_code values
missing_alc_fellows = (
    fellows_clean[
        fellows_clean["alc_code"].isnull()
    ]
)

missing_alc_employer = (
    employer_clean[
        employer_clean["alc_code"].isnull()
    ]
)


# Display affected fellows records
display(missing_alc_fellows.head())


# Display affected employer engagement records
display(missing_alc_employer.head())


# Assess whether alc_code values can be inferred
# using related categorical fields
alc_inference_check = (
    fellows_clean.groupby(
        ["state", "track"]
    )["alc_code"]
    .nunique()
    .reset_index()
)


# Display grouped ALC distribution samples
display(alc_inference_check.head(10))

**Sequential ALC Code Repair**

The ALC weekly logs were reviewed separately before validating the missing `alc_code` values across datasets.

Some weekly operational records contained missing ALC codes despite following continuous weekly sequences.

These missing values were therefore repaired using forward-fill and backward-fill logic before continuing with the mapping validation process.

In [ ]:
# Sort operational logs sequentially
alc_logs_clean = (
    alc_logs_clean
    .sort_values(
        by=["state", "week_number"]
    )
)


# Fill missing ALC codes sequentially
alc_logs_clean["alc_code"] = (
    alc_logs_clean["alc_code"]
    .ffill()
    .bfill()
)


# Validate remaining missing values
print(
    "Remaining missing alc_code values:",
    alc_logs_clean["alc_code"].isnull().sum()
)

In [ ]:
# Compare state values across all datasets

# Extract unique states from each dataset
fellows_states = set(
    fellows_clean["state"]
    .dropna()
    .str.strip()
    .str.title()
    .unique()
)

employer_states = set(
    employer_clean["state"]
    .dropna()
    .str.strip()
    .str.title()
    .unique()
)

alc_states = set(
    alc_logs_clean["state"]
    .dropna()
    .str.strip()
    .str.title()
    .unique()
)


# Build comparison summary
state_comparison = pd.DataFrame({

    "Dataset": [
        "Fellows Cohort",
        "Employer Engagement",
        "ALC Weekly Logs"
    ],

    "Unique States": [
        len(fellows_states),
        len(employer_states),
        len(alc_states)
    ]

})


# Display state summary
display(
    state_comparison.style
    .background_gradient(
        cmap="Blues",
        subset=["Unique States"]
    )
)


# Identify states missing across datasets
fellows_not_in_alc = sorted(
    fellows_states - alc_states
)

employer_not_in_alc = sorted(
    employer_states - alc_states
)


# Display unmatched states
print(
    "Fellows states not found in ALC logs:",
    fellows_not_in_alc
)

print(
    "Employer states not found in ALC logs:",
    employer_not_in_alc
)

**Mapping Validation**

After validating state coverage across datasets, additional mapping checks were performed to determine whether missing `alc_code` values could be inferred reliably using operational and enrollment-level features.

In [ ]:
# Create enrollment week feature
fellows_clean["enrollment_week"] = (
    fellows_clean["enrollment_date"]
    .dt.isocalendar()
    .week
)


# Build refined ALC mapping validation table
alc_mapping_validation = (
    fellows_clean.groupby(
        [
            "state",
            "track",
            "cohort_number",
            "enrollment_week"
        ]
    )["alc_code"]
    .agg(
        unique_alc_count=lambda x: x.dropna().nunique(),
        alc_codes=lambda x: sorted(x.dropna().unique())
    )
    .reset_index()
)


# Identify only ambiguous mappings
ambiguous_alc_mappings = (
    alc_mapping_validation[
        alc_mapping_validation["unique_alc_count"] > 1
    ]
)


# Display ambiguous combinations
display(ambiguous_alc_mappings)


# Display ambiguity count
print(
    f"Ambiguous combinations found: "
    f"{len(ambiguous_alc_mappings)}"
)

**Observations**

Most records matched clearly to a single ALC code after combining:
- state
- track
- cohort number
- enrollment week

This means the missing ALC codes could be filled safely for most records.

However, 9 combinations still matched to more than one ALC code. These records were left unchanged to avoid assigning the wrong ALC.

In [ ]:
# PASS 1: Build the mapping dict (state + track + cohort + enrollment_week)
map_1 = (
    fellows_clean[fellows_clean["alc_code"].notna()]
    .groupby(["state", "track", "cohort_number", "enrollment_week"])["alc_code"]
    .agg(lambda x: x.iloc[0] if x.nunique() == 1 else np.nan)
    .dropna()
    .to_dict()
)

# Apply Pass 1
fellows_clean["alc_code"] = fellows_clean.apply(
    lambda row: map_1.get(
        (row["state"], row["track"], row["cohort_number"], row["enrollment_week"]),
        row["alc_code"]
    ) if pd.isna(row["alc_code"]) else row["alc_code"],
    axis=1
)

# PASS 2: state + track + cohort_number
map_2 = (
    fellows_clean[fellows_clean["alc_code"].notna()]
    .groupby(["state", "track", "cohort_number"])["alc_code"]
    .agg(lambda x: x.iloc[0] if x.nunique() == 1 else np.nan)
    .dropna()
    .to_dict()
)

fellows_clean["alc_code"] = fellows_clean.apply(
    lambda row: map_2.get(
        (row["state"], row["track"], row["cohort_number"]),
        row["alc_code"]
    ) if pd.isna(row["alc_code"]) else row["alc_code"],
    axis=1
)

# PASS 3: state + track
map_3 = (
    fellows_clean[fellows_clean["alc_code"].notna()]
    .groupby(["state", "track"])["alc_code"]
    .agg(lambda x: x.iloc[0] if x.nunique() == 1 else np.nan)
    .dropna()
    .to_dict()
)

fellows_clean["alc_code"] = fellows_clean.apply(
    lambda row: map_3.get(
        (row["state"], row["track"]),
        row["alc_code"]
    ) if pd.isna(row["alc_code"]) else row["alc_code"],
    axis=1
)

In [ ]:
print("Rows before:", len(fellows))       # original
print("Rows after:", len(fellows_clean))  # should match

In [ ]:
# FINAL NULL VALIDATION

# Build final missing value summary
final_missing_summary = pd.DataFrame({

    "Dataset": [
        "Fellows Cohort",
        "Employer Engagement",
        "ALC Weekly Logs"
    ],

    "Remaining Missing alc_code": [
        fellows_clean["alc_code"].isnull().sum(),
        employer_clean["alc_code"].isnull().sum(),
        alc_logs_clean["alc_code"].isnull().sum()
    ]

})


# Display final validation summary
display(
    final_missing_summary.style
    .background_gradient(
        cmap="Oranges",
        subset=["Remaining Missing alc_code"]
    )
)

**Filling Summary**

The missing `alc_code` values were filled using a staged matching process.

The process started by repairing the missing ALC codes inside the ALC weekly logs using forward-fill and backward-fill logic. This created a cleaner operational reference table for downstream validation.

Next, state values were compared across all datasets to confirm structural consistency and identify states missing from the operational logs.

Additional mapping checks were then performed using:
- state
- track
- cohort number
- enrollment week

These combinations were used to test whether records mapped consistently to a single ALC code.

The filling process was applied in multiple stages:
- First using `state + track + cohort_number + enrollment_week`
- Then using `state + track + cohort_number`
- Finally using `state + track`

At each stage, records were only filled when the combination mapped to one unique ALC code. Any combinations linked to multiple ALC codes were treated as ambiguous and were not filled.

This approach allowed most missing values to be resolved safely while preserving the integrity of records that could not be matched confidently.

In [ ]:
# DISPLAY REMAINING UNRESOLVED RECORDS

# Filter records with unresolved ALC codes
remaining_unresolved = (
    fellows_clean[
        fellows_clean["alc_code"].isnull()
    ]
)


# Select key validation columns
remaining_unresolved = (
    remaining_unresolved[
        [
            "fellow_id",
            "state",
            "track",
            "cohort_number",
            "enrollment_week",
            "completion_status",
            "alc_code"
        ]
    ]
)


# Display unresolved records
display(remaining_unresolved)


# Display unresolved record count
print(
    f"Remaining unresolved records: "
    f"{len(remaining_unresolved)}"
)

**Remaining Unresolved Records**

The final unresolved records fell into two groups.

The first group involved states such as Ekiti that did not appear in the operational ALC weekly logs. These records could not be validated against an operational reference source.

The second group involved states such as Kaduna, Osun, Yobe, Plateau, Benue, Jigawa, and Adamawa. These states existed in the operational logs but mapped to multiple valid ALC codes across different operational patterns.

Because these records could not be matched to one unique ALC code confidently, they were left unchanged.

In [ ]:
# CREATE ACTIVE WORKING TABLES

# Create working copies for downstream processing
fellows_working = fellows_clean.copy()

reflection_working = reflection_clean.copy()

employer_working = employer_clean.copy()

alc_logs_working = alc_logs_clean.copy()


# Display working table shapes
print("Fellows Cohort:", fellows_working.shape)

print("Reflection Survey:", reflection_working.shape)

print("Employer Engagement:", employer_working.shape)

print("ALC Weekly Logs:", alc_logs_working.shape)


# Preview working datasets
display(fellows_working.head())

display(reflection_working.head())

display(employer_working.head())

display(alc_logs_working.head())

## 4.4 Manage Duplicates

Duplicate checks were performed separately for each dataset using operational uniqueness rules instead of full-row matching.

Each dataset represents a different type of activity, so duplicate logic was based on the fields expected to uniquely identify a valid operational record.

**Fellows Cohort**

The fellows cohort dataset represents enrollment-level records, meaning each `fellow_id` should appear only once.

Repeated `fellow_id` values were reviewed to determine whether they represented exact duplicate rows or conflicting enrollment records before removal.

In [ ]:
# Check repeated fellow IDs
fellows_duplicate_check = (
    fellows_working[
        fellows_working.duplicated(
            subset=["fellow_id"],
            keep=False
        )
    ]
    .sort_values("fellow_id")
)


# Display duplicate fellow records
display(fellows_duplicate_check)


# Count duplicate fellow IDs
duplicate_fellow_count = (
    fellows_working["fellow_id"]
    .duplicated()
    .sum()
)


print(
    f"Duplicate fellow records found: "
    f"{duplicate_fellow_count}"
)


# Remove duplicate fellow records
fellows_deduplicated = (
    fellows_working
    .sort_values(
        by=[
            "alc_code",
            "certification_status"
        ],
        ascending=False
    )
    .drop_duplicates(
        subset=["fellow_id"],
        keep="first"
    )
)


# Validate row count after deduplication
print(
    f"Rows before deduplication: "
    f"{len(fellows_working)}"
)

print(
    f"Rows after deduplication: "
    f"{len(fellows_deduplicated)}"
)

**Reflection Survey**

The reflection survey dataset was assessed using the combination of `fellow_id + week` as the expected uniqueness rule since fellows can submit surveys across multiple weeks.

Only repeated submissions within the same week were treated as duplicates. Where multiple weekly submissions existed, the latest survey response was retained using the `response_timestamp` field before older entries were removed.

In [ ]:
# Check repeated weekly survey submissions
reflection_duplicate_check = (
    reflection_working[
        reflection_working.duplicated(
            subset=[
                "fellow_id",
                "week"
            ],
            keep=False
        )
    ]
    .sort_values(
        by=[
            "fellow_id",
            "week",
            "response_timestamp"
        ]
    )
)


# Display duplicate survey records
display(reflection_duplicate_check)


# Count duplicate survey submissions
duplicate_survey_count = (
    reflection_working.duplicated(
        subset=[
            "fellow_id",
            "week"
        ]
    ).sum()
)


print(
    f"Duplicate survey records found: "
    f"{duplicate_survey_count}"
)


# Retain latest survey submission
reflection_deduplicated = (
    reflection_working
    .sort_values(
        by="response_timestamp",
        ascending=False
    )
    .drop_duplicates(
        subset=[
            "fellow_id",
            "week"
        ],
        keep="first"
    )
)


# Validate row count after deduplication
print(
    f"Rows before deduplication: "
    f"{len(reflection_working)}"
)

print(
    f"Rows after deduplication: "
    f"{len(reflection_deduplicated)}"
)

**Employer Engagement**

The employer engagement dataset was assessed using the combination of `employer_name + engagement_type + engagement_date + alc_code` as the operational event identifier.

Repeated operational events using the same combination were reviewed as potential duplicates. Where duplicate events existed, the most complete operational record was retained before confirmed duplicates were removed.

In [ ]:

# Check exact full-row duplicates
employer_duplicate_check = (
    employer_working[
        employer_working.duplicated(
            keep=False
        )
    ]
    .sort_values(
        by=[
            "employer_name",
            "engagement_date"
        ]
    )
)


# Display duplicate employer records
display(employer_duplicate_check)


# Count exact duplicate rows
duplicate_employer_count = (
    employer_working.duplicated().sum()
)


print(
    f"Exact duplicate employer records found: "
    f"{duplicate_employer_count}"
)


# Remove exact full-row duplicates only
employer_deduplicated = (
    employer_working
    .drop_duplicates()
)


# Validate row count after deduplication
print(
    f"Rows before deduplication: "
    f"{len(employer_working)}"
)

print(
    f"Rows after deduplication: "
    f"{len(employer_deduplicated)}"
)

**ALC Weekly Logs**

The ALC weekly logs dataset was reviewed using `alc_code + week_number` as the expected operational reporting key since each ALC should normally submit one attendance log per week.

Repeated weekly logs were reviewed carefully to identify exact duplicates and conflicting operational records. Where duplicate rows were confirmed, the most reliable operational record was retained before confirmed duplicates were removed.

In [ ]:
# Check repeated ALC weekly logs
alc_duplicate_check = (
    alc_logs_working[
        alc_logs_working.duplicated(
            subset=[
                "alc_code",
                "week_number"
            ],
            keep=False
        )
    ]
    .sort_values(
        by=[
            "alc_code",
            "week_number"
        ]
    )
)


# Display duplicate ALC logs
display(alc_duplicate_check)


# Count duplicate weekly logs
duplicate_alc_count = (
    alc_logs_working.duplicated(
        subset=[
            "alc_code",
            "week_number"
        ]
    ).sum()
)


print(
    f"Duplicate ALC weekly logs found: "
    f"{duplicate_alc_count}"
)


# Retain most complete operational log
alc_logs_deduplicated = (
    alc_logs_working
    .sort_values(
        by=[
            "sessions_held",
            "fellows_present"
        ],
        ascending=False
    )
    .drop_duplicates(
        subset=[
            "alc_code",
            "week_number"
        ],
        keep="first"
    )
)


# Validate row count after deduplication
print(
    f"Rows before deduplication: "
    f"{len(alc_logs_working)}"
)

print(
    f"Rows after deduplication: "
    f"{len(alc_logs_deduplicated)}"
)

## 5. Feature Engineering & Analytical Preparation

### 5.1 Temporal and Geographic Feature Preparation

Additional temporal and geographic features were prepared to support downstream aggregation, segmentation, filtering, and regional analysis.

Enrollment periods were standardized into reusable analytical time windows, while geopolitical zones were propagated using validated state-level mappings to ensure consistent regional analysis across datasets.

In [ ]:
# Create enrollment month
fellows_deduplicated["enrollment_month"] = (
    fellows_deduplicated["enrollment_date"]
    .dt.to_period("M")
    .astype(str)
)


# Create enrollment week
fellows_deduplicated["enrollment_week"] = (
    fellows_deduplicated["enrollment_date"]
    .dt.isocalendar()
    .week
)


# Preview engineered temporal features
display(
    fellows_deduplicated[
        [
            "enrollment_date",
            "enrollment_month",
            "enrollment_week"
        ]
    ].head()
)

In [ ]:
# GEOPOLITICAL ZONE STANDARDIZATION

# Create validated state-to-zone reference table
state_zone_mapping = (
    employer_deduplicated[
        [
            "state",
            "geopolitical_zone"
        ]
    ]
    .dropna()
    .drop_duplicates()
)


# Remove existing geopolitical zone column if present
fellows_deduplicated = (
    fellows_deduplicated.drop(
        columns=["geopolitical_zone"],
        errors="ignore"
    )
)


# Merge geopolitical zones into fellows dataset
fellows_deduplicated = (
    fellows_deduplicated.merge(
        state_zone_mapping,
        on="state",
        how="left"
    )
)


# Validate missing geopolitical zones
print(
    "Missing geopolitical zones:",
    fellows_deduplicated[
        "geopolitical_zone"
    ].isnull().sum()
)

### 5.2 At-Risk Fellow Features

Additional engagement and duration features were created to support the at-risk fellow logic used in the operational analysis.

These features help identify fellows with incomplete progress, low recent engagement, and extended enrollment duration.

In [ ]:
# =========================================================
# STAGED AT-RISK FELLOW FEATURE ENGINEERING
# =========================================================

# Get latest survey response week per fellow
latest_response = (
    reflection_deduplicated.groupby(
        "fellow_id"
    )["week"]
    .max()
    .reset_index()
)

latest_response.columns = [
    "fellow_id",
    "latest_response_week"
]


# Remove existing latest response column if present
fellows_deduplicated = (
    fellows_deduplicated.drop(
        columns=["latest_response_week"],
        errors="ignore"
    )
)


# Merge latest response week into fellows dataset
fellows_deduplicated = (
    fellows_deduplicated.merge(
        latest_response,
        on="fellow_id",
        how="left"
    )
)


# Define current operational week
current_week = (
    reflection_deduplicated["week"]
    .max()
)


# Calculate weeks since last survey response
fellows_deduplicated["weeks_since_last_response"] = (

    current_week

    -

    fellows_deduplicated[
        "latest_response_week"
    ]
)


# Calculate enrollment duration in weeks
current_date = (
    fellows_deduplicated["enrollment_date"]
    .max()
)

fellows_deduplicated["enrollment_duration_weeks"] = (

    (
        current_date
        -
        fellows_deduplicated[
            "enrollment_date"
        ]
    )

    .dt.days // 7
)


# =========================================================
# STAGED RISK EVALUATION
# =========================================================

# Step 1:
# Identify fellows enrolled for more than 3 weeks
fellows_deduplicated["eligible_for_risk_review"] = np.where(

    fellows_deduplicated[
        "enrollment_duration_weeks"
    ] > 3,

    1,
    0
)


# Step 2:
# Identify fellows inactive for more than 2 weeks
fellows_deduplicated["inactive_engagement_flag"] = np.where(

    fellows_deduplicated[
        "weeks_since_last_response"
    ] > 1,

    1,
    0
)


# Step 3:
# Create final at-risk flag
fellows_deduplicated["is_at_risk"] = np.where(

    (
        fellows_deduplicated[
            "eligible_for_risk_review"
        ] == 1
    )

    &

    (
        fellows_deduplicated[
            "inactive_engagement_flag"
        ] == 1
    )

    &

    (
        fellows_deduplicated[
            "completion_status"
        ] == "incomplete"
    ),

    1,
    0
)


# Preview updated at-risk features
display(
    fellows_deduplicated[
        [
            "fellow_id",
            "completion_status",
            "latest_response_week",
            "weeks_since_last_response",
            "enrollment_duration_weeks",
            "eligible_for_risk_review",
            "inactive_engagement_flag",
            "is_at_risk"
        ]
    ].head()
)

Additional engagement and duration features were created to support the operational at-risk fellow definition used in the downstream analysis.

The at-risk evaluation first identified fellows who had remained enrolled in the program for more than three weeks to avoid prematurely assessing newly onboarded participants.

Among those fellows, a fellow was classified as at-risk if:
- the completion status remained incomplete,
- and no reflection survey response had been recorded within the last two weeks.

The shorter enrollment threshold was selected to support earlier disengagement detection and more proactive intervention monitoring.

### 5.3 Survey Engagement Features

Survey engagement metrics were created to support engagement trajectory analysis and participation monitoring across fellows.

These features help measure response activity, consistency, and average engagement performance over time.

In [ ]:
# =========================================================
# SURVEY ENGAGEMENT FEATURE ENGINEERING
# =========================================================

# Create engagement summary metrics
survey_engagement_features = (
    reflection_deduplicated.groupby(
        "fellow_id"
    )
    .agg(

        total_survey_responses=(
            "week",
            "count"
        ),

        average_survey_score=(
            "survey_score",
            "mean"
        )

    )
    .reset_index()
)


# Merge engagement metrics into fellows dataset
fellows_deduplicated = (
    fellows_deduplicated.merge(
        survey_engagement_features,
        on="fellow_id",
        how="left"
    )
)


# Preview engagement features
display(
    fellows_deduplicated[
        [
            "fellow_id",
            "total_survey_responses",
            "average_survey_score",
            "latest_response_week"
        ]
    ].head()
)

### 5.4 Operational Flags

Simple operational indicator flags were created to support dashboard filtering, statistical testing, and categorical performance analysis.

These flags convert operational completion and certification outcomes into reusable binary analytical variables.

In [ ]:
# Create completion flag
fellows_deduplicated["completion_flag"] = np.where(
    fellows_deduplicated["completion_status"] == "complete",
    1,
    0
)


# Create certification flag
fellows_deduplicated["certification_flag"] = np.where(
    fellows_deduplicated["certification_status"] == "certified",
    1,
    0
)


# Preview operational flags
display(
    fellows_deduplicated[
        [
            "fellow_id",
            "completion_status",
            "completion_flag",
            "certification_status",
            "certification_flag"
        ]
    ].head()
)

### 5.5 Attendance Participation Ratio

Attendance participation ratios were created to support ALC-level operational analysis, performance comparison, and dashboard ranking.

The ratio measures average fellow attendance relative to the number of sessions conducted across reporting periods.

In [ ]:
# Create attendance participation ratio
alc_logs_deduplicated["attendance_participation_ratio"] = (

    alc_logs_deduplicated["fellows_present"]

    /

    alc_logs_deduplicated["sessions_held"]

)


# Preview participation ratio
display(
    alc_logs_deduplicated[
        [
            "alc_code",
            "week_number",
            "fellows_present",
            "sessions_held",
            "attendance_participation_ratio"
        ]
    ].head()
)

## 6. Exploratory Analysis & Business Insights

### 6.1 AT-RISK FELLOW SEGMENT ANALYSIS

In [ ]:
# Create updated at-risk subset
at_risk_fellows = (
    fellows_deduplicated[
        fellows_deduplicated["is_at_risk"] == 1
    ]
)


# =========================================================
# SEGMENT SIZE
# =========================================================

# Calculate at-risk segment size
at_risk_summary = pd.DataFrame({

    "Metric": [
        "Total Fellows",
        "At-Risk Fellows",
        "At-Risk Percentage"
    ],

    "Value": [

        len(fellows_deduplicated),

        len(at_risk_fellows),

        round(
            (
                len(at_risk_fellows)
                /
                len(fellows_deduplicated)
            ) * 100,
            2
        )
    ]
})


# Display segment summary
display(at_risk_summary)


# =========================================================
# ZONE BREAKDOWN
# =========================================================

# Analyze at-risk fellows by geopolitical zone
at_risk_by_zone = (
    at_risk_fellows.groupby(
        "geopolitical_zone"
    )
    .size()
    .reset_index(name="at_risk_count")
    .sort_values(
        by="at_risk_count",
        ascending=False
    )
)


# Display zone breakdown
display(at_risk_by_zone)


# =========================================================
# TRACK BREAKDOWN
# =========================================================

# Analyze at-risk fellows by track
at_risk_by_track = (
    at_risk_fellows.groupby(
        "track"
    )
    .size()
    .reset_index(name="at_risk_count")
    .sort_values(
        by="at_risk_count",
        ascending=False
    )
)


# Display track breakdown
display(at_risk_by_track)


# =========================================================
# ENGAGEMENT TRAJECTORY
# =========================================================

# Analyze engagement trajectory metrics
at_risk_engagement = (
    at_risk_fellows[
        [
            "total_survey_responses",
            "average_survey_score",
            "weeks_since_last_response",
            "enrollment_duration_weeks"
        ]
    ]
    .describe()
    .round(2)
)


# Display engagement trajectory summary
display(at_risk_engagement)


# =========================================================
# DISPLAY AT-RISK FELLOWS
# =========================================================

# Preview at-risk fellows
display(
    at_risk_fellows[
        [
            "fellow_id",
            "state",
            "geopolitical_zone",
            "track",
            "completion_status",
            "eligible_for_risk_review",
            "inactive_engagement_flag",
            "weeks_since_last_response",
            "enrollment_duration_weeks",
            "total_survey_responses",
            "average_survey_score"
        ]
    ]
    .sort_values(
        by=[
            "weeks_since_last_response",
            "enrollment_duration_weeks"
        ],
        ascending=False
    )
)

**At-Risk Fellow Criteria**

A fellow was classified as at-risk using a staged operational assessment process.

The evaluation first identified fellows who had remained enrolled in the program for more than three weeks to avoid prematurely assessing newly onboarded participants.

Among those fellows, additional risk conditions were evaluated:
- the completion status remained incomplete,
- and no reflection survey response had been submitted within the last two weeks.

This staged logic was designed to support earlier disengagement detection while maintaining a realistic onboarding adjustment window.

**At-Risk Fellow Analysis Summary**

The analysis identified 23 at-risk fellows representing approximately 2% of the total fellow population.

The segment was concentrated primarily within the North-East and North-West geopolitical zones, which together accounted for more than half of the identified at-risk fellows. Track-level analysis showed that AI/ML and Cybersecurity recorded the highest concentration of at-risk participants.

Engagement trajectory analysis revealed that most at-risk fellows had missed at least two consecutive reflection survey cycles despite remaining enrolled in the program for extended periods. The average enrollment duration for the segment was approximately 23 weeks, suggesting that many affected fellows had remained in the program long enough for disengagement patterns to become operationally significant.

Although the segment maintained moderate average survey scores overall, the combination of incomplete program status, declining recent engagement activity, and prolonged enrollment duration suggests a need for targeted intervention and follow-up support.

### 6.2 Operational Time Scope Validation

Before conducting comparative analysis and statistical testing, the operational time scope of the datasets was validated to understand the coverage period of the analysis.

This step is important because analytical conclusions may vary across operational cycles, cohorts, or reporting periods. Establishing the temporal scope first helps ensure that later comparisons, trends, and statistical findings are interpreted within the correct business context.

In [ ]:
# =========================================================
# OPERATIONAL TIME SCOPE VALIDATION
# =========================================================

# Extract operational years
fellows_deduplicated["enrollment_year"] = (
    fellows_deduplicated["enrollment_date"]
    .dt.year
)

employer_deduplicated["engagement_year"] = (
    employer_deduplicated["engagement_date"]
    .dt.year
)


# =========================================================
# OVERALL DATE RANGE SUMMARY
# =========================================================

# Build overall operational date summary
time_scope_summary = pd.DataFrame({

    "Dataset": [
        "Fellows Cohort",
        "Employer Engagement"
    ],

    "Start Date": [

        fellows_deduplicated[
            "enrollment_date"
        ].min(),

        employer_deduplicated[
            "engagement_date"
        ].min()
    ],

    "End Date": [

        fellows_deduplicated[
            "enrollment_date"
        ].max(),

        employer_deduplicated[
            "engagement_date"
        ].max()
    ],

    "Years Covered": [

        sorted(
            fellows_deduplicated[
                "enrollment_year"
            ].unique()
        ),

        sorted(
            employer_deduplicated[
                "engagement_year"
            ].unique()
        )
    ]
})


# Display operational time scope summary
display(time_scope_summary)


# =========================================================
# YEARLY RECORD DISTRIBUTION
# =========================================================

# Fellows yearly distribution
fellows_yearly_distribution = (
    fellows_deduplicated.groupby(
        "enrollment_year"
    )
    .size()
    .reset_index(name="fellow_count")
)


# Employer yearly distribution
employer_yearly_distribution = (
    employer_deduplicated.groupby(
        "engagement_year"
    )
    .size()
    .reset_index(name="engagement_count")
)


# Display yearly breakdowns
display(fellows_yearly_distribution)

display(employer_yearly_distribution)